In [1]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy

In [2]:
import os, sys, json, subprocess
from pathlib import Path
from collections import Counter
import torch, transformers

ROOT = Path('/content')
ORIGINAL = ROOT / 'PIGuard'
TEAM = ROOT / 'piguard-extension'
RESULTS = ROOT / 'korean_evaluation_results'
RESULTS.mkdir(exist_ok=True)
if not ORIGINAL.exists():
    subprocess.run(['git', 'clone', 'https://github.com/leolee99/PIGuard.git', str(ORIGINAL)], check=True)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available())
print('Original code revision:')
subprocess.run(['git', '-C', str(ORIGINAL), 'rev-parse', 'HEAD'], check=True)

PyTorch: 2.14.1+cpu
Transformers: 5.22.1
CUDA: False
Original code revision:


CompletedProcess(args=['git', '-C', '/content/PIGuard', 'rev-parse', 'HEAD'], returncode=0)

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = 'leolee99/PIGuard'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)
classifier = pipeline('text-classification', model=model, tokenizer=tokenizer, truncation=True, device=-1)
print('Model:', model_name)
print('Model revision:', getattr(model.config, '_commit_hash', None))
print('Labels:', model.config.id2label)

config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

The repository leolee99/PIGuard contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/leolee99/PIGuard .
 You can inspect the repository content at https://hf.co/leolee99/PIGuard.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


modeling_piguard.py:   0%|          | 0.00/973 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/leolee99/PIGuard:
- modeling_piguard.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B /   132B            

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model: leolee99/PIGuard
Model revision: None
Labels: {0: 'benign', 1: 'injection'}


In [4]:
RUN_ORIGINAL = False  # 필요할 때 True로 변경
if RUN_ORIGINAL:
    sys.path.insert(0, str(ORIGINAL))
    from eval_hf import wildguard_eval, BIPIA_eval, NotInject_eval
    dataset_root = str(ORIGINAL / 'datasets')
    wild = wildguard_eval(classifier, dataset_root)
    bipia, text_acc, code_acc = BIPIA_eval(classifier, dataset_root)
    notinject, one, two, three = NotInject_eval(classifier, dataset_root)
    print('Macro average:', (wild + bipia + notinject) / 3)

In [8]:
from google.colab import files
names = ['Korean_NotInject_all.json', 'Korean_Benign_Additional_120.json']
def locate(name):
    candidates = [ROOT / name, ORIGINAL / name, TEAM / 'extension/korean_testset' / name, Path.cwd() / name]
    return next((p for p in candidates if p.is_file()), None)
if any(locate(name) is None for name in names):
    files.upload()
def load_normal(name, expected):
    path = locate(name)
    if path is None:
        raise FileNotFoundError(name)
    data = json.loads(path.read_text(encoding='utf-8'))
    assert len(data) == expected
    assert all(isinstance(x['prompt'], str) and x['target_class'] == 'benign' for x in data)
    return data
benign_150 = load_normal(names[0], 150)
benign_120 = load_normal(names[1], 120)
print('정상 데이터:', len(benign_150) + len(benign_120))
print('기존 키워드 분포:', Counter(x['keyword_count'] for x in benign_150))
print('추가 유형 분포:', Counter(x['category'] for x in benign_120))

Saving Korean_Benign_Additional_120.json to Korean_Benign_Additional_120.json
정상 데이터: 270
기존 키워드 분포: Counter({1: 50, 2: 50, 3: 50})
추가 유형 분포: Counter({'daily_information': 30, 'work_email': 30, 'programming_security': 30, 'document_analysis': 30})


In [9]:
def acc_compute(model, prompts, target_class="benign", name=""):
    correct = 0
    total = len(prompts)

    predictions = model(prompts)

    for prediction in predictions:
        predicted_label = prediction["label"]

        if predicted_label == target_class:
            correct += 1

    accuracy = correct / total if total > 0 else 0

    print(f"{name} Accuracy: {accuracy:.4f}")
    print(f"{name} 정확도: {accuracy * 100:.2f}%")

    return accuracy

In [10]:
# 기존 정상 데이터 150개
prompts_150 = [item["prompt"] for item in benign_150]

# 추가 정상 데이터 120개
prompts_120 = [item["prompt"] for item in benign_120]

print("========== 한국어 정상 데이터 평가 ==========")

accuracy_150 = acc_compute(
    classifier,
    prompts_150,
    target_class="benign",
    name="Korean Benign 150"
)

accuracy_120 = acc_compute(
    classifier,
    prompts_120,
    target_class="benign",
    name="Korean Benign Additional 120"
)

# 전체 정확도 계산
accuracy_270 = (
    accuracy_150 * len(prompts_150)
    + accuracy_120 * len(prompts_120)
) / (len(prompts_150) + len(prompts_120))

print("\n========== 최종 평가 결과 ==========")
print(f"기존 정상 데이터: {len(prompts_150)}개")
print(f"기존 정상 데이터 정확도: {accuracy_150 * 100:.2f}%")

print(f"추가 정상 데이터: {len(prompts_120)}개")
print(f"추가 정상 데이터 정확도: {accuracy_120 * 100:.2f}%")

print(f"전체 정상 데이터: {len(prompts_150) + len(prompts_120)}개")
print(f"전체 정상 데이터 정확도: {accuracy_270 * 100:.2f}%")
print(f"정상 데이터 오탐률(FPR): {(1-accuracy_270) * 100:.2f}%")

========== 한국어 정상 데이터 평가 ==========
Korean Benign 150 Accuracy: 1.0000
Korean Benign 150 정확도: 100.00%
Korean Benign Additional 120 Accuracy: 1.0000
Korean Benign Additional 120 정확도: 100.00%

========== 최종 평가 결과 ==========
기존 정상 데이터: 150개
기존 정상 데이터 정확도: 100.00%
추가 정상 데이터: 120개
추가 정상 데이터 정확도: 100.00%
전체 정상 데이터: 270개
전체 정상 데이터 정확도: 100.00%
정상 데이터 오탐률(FPR): 0.00%


In [11]:
benign_results = {
    'model': model_name, 'language': 'Korean', 'evaluation': 'benign',
    'original_benign': {'total': len(benign_150), 'accuracy': accuracy_150},
    'additional_benign': {'total': len(benign_120), 'accuracy': accuracy_120},
    'overall': {'total': 270, 'correct': round(270 * accuracy_270),
                'incorrect': 270 - round(270 * accuracy_270),
                'accuracy': accuracy_270, 'false_positive_rate': 1 - accuracy_270}
}
(RESULTS / 'Korean_Benign_Results.json').write_text(json.dumps(benign_results, ensure_ascii=False, indent=2), encoding='utf-8')

347

In [12]:
# 기존 실행 결과의 집계값 (재실행 아님)
attack_record = {
    'direct_override': {'total': 30, 'correct': 13},
    'system_prompt_extraction': {'total': 30, 'correct': 4},
    'role_and_policy_override': {'total': 30, 'correct': 15},
    'indirect_document_injection': {'total': 30, 'correct': 8},
}
attack_total = sum(x['total'] for x in attack_record.values())
attack_correct = sum(x['correct'] for x in attack_record.values())
for x in attack_record.values():
    x['accuracy'] = x['correct'] / x['total']
print('기존 공격 평가 집계:', attack_correct, '/', attack_total)
print(f'기존 탐지율: {attack_correct / attack_total:.2%}')
(RESULTS / 'Korean_Attack_Historical_Results.json').write_text(json.dumps(attack_record, indent=2), encoding='utf-8')

기존 공격 평가 집계: 40 / 120
기존 탐지율: 33.33%


414

In [14]:
TN, FP, TP, FN = 270, 0, attack_correct, attack_total - attack_correct
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if TP + FP else 0
recall = TP / (TP + FN)
specificity = TN / (TN + FP)
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
historical = {
    'model': 'leolee99/PIGuard', 'language': 'Korean',
    'provenance': 'historical notebook evaluation; aggregate recomputation',
    'total_samples': 390, 'benign_samples': 270, 'attack_samples': 120,
    'confusion_matrix': {'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN},
    'metrics': {'accuracy': accuracy, 'precision': precision, 'recall': recall,
                'f1_score': f1, 'balanced_accuracy': (recall + specificity) / 2,
                'false_positive_rate': FP / (FP + TN),
                'false_negative_rate': FN / (FN + TP)}
}
for key, val in historical['metrics'].items():
    print(f'{key}: {val:.2%}')
(RESULTS / 'Korean_Final_Historical_Results.json').write_text(json.dumps(historical, indent=2), encoding='utf-8');

accuracy: 79.49%
precision: 100.00%
recall: 33.33%
f1_score: 50.00%
balanced_accuracy: 66.67%
false_positive_rate: 0.00%
false_negative_rate: 66.67%
